<a href="https://colab.research.google.com/github/KevinKenya/nairobi-connector-open-source/blob/main/nairobiOs02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
eoinamoore_historical_nba_data_and_player_box_scores_path = kagglehub.dataset_download('eoinamoore/historical-nba-data-and-player-box-scores')

print('Data source import complete.')

100%|██████████| 1.03G/1.03G [00:11<00:00, 101MB/s]

Extracting files...


Data source import complete.


In [2]:
!pip install nairobi_os -vv

Using pip 24.1.2 from /usr/local/lib/python3.12/dist-packages/pip (python 3.12)
Non-user install because site-packages writeable
Created temporary directory: /tmp/pip-build-tracker-ku_pifmx
Initialized build tracking at /tmp/pip-build-tracker-ku_pifmx
Created build tracker: /tmp/pip-build-tracker-ku_pifmx
Entered build tracker: /tmp/pip-build-tracker-ku_pifmx
Created temporary directory: /tmp/pip-install-360jwjwx
Created temporary directory: /tmp/pip-ephem-wheel-cache-2utu2705
1 location(s) to search for versions of nairobi-os:
* https://pypi.org/simple/nairobi-os/
Fetching project page and analyzing links: https://pypi.org/simple/nairobi-os/
Getting page https://pypi.org/simple/nairobi-os/
Found index url https://pypi.org/simple/
Looking up "https://pypi.org/simple/nairobi-os/" in the cache
Request header has "max_age" as 0, cache bypassed
No cache entry available
Starting new HTTPS connection (1): pypi.org:443
https://pypi.org:443 "GET /simple/nairobi-os/ HTTP/1.1" 200 775
Updating c

In [3]:
import os
import subprocess
from pathlib import Path
import nairobi_os

print("🛠 1. Installing D-Bus Infrastructure...")
subprocess.run(["apt-get", "update"], capture_output=True)
subprocess.run(["apt-get", "install", "-y", "dbus-x11"], capture_output=True)

print("🔌 2. Initializing D-Bus Session...")
dbus_out = subprocess.check_output(["dbus-launch"]).decode()
for line in dbus_out.splitlines():
    if "=" in line:
        k, v = line.split("=", 1)
        os.environ[k] = v.replace(";", "").replace("'", "").replace('"', '')

print("🔐 3. Granting Executable Permissions...")
bin_path = Path(nairobi_os.__file__).parent / "bin" / "nairobi-axum-refinery"
os.chmod(bin_path, 0o755)

print("🔥 4. Igniting the Heavy Iron (v0.2.1)...")
try:
    nairobi_os.start_refinery()
    print("✅ EMPIRE ONLINE")
except Exception as e:
    print(f"\n💥 Ignition Failed. Dumping Forensic Logs:")
    os.system("cat ~/.nairobi_refinery.log")

🛠 1. Installing D-Bus Infrastructure...
🔌 2. Initializing D-Bus Session...
🔐 3. Granting Executable Permissions...
🔥 4. Igniting the Heavy Iron (v0.2.1)...
✅ EMPIRE ONLINE


In [4]:
import pandas as pd
import numpy as np
import nairobi_os
import json
import os
import stat
from pathlib import Path

# Create a synthetic dataset
df = pd.DataFrame({
    'target_col': np.random.rand(100),
    'col1': np.random.rand(100),
    'col2': np.random.rand(100)
})
df.to_csv('dataset.csv', index=False)
print("✅ Created synthetic dataset: dataset.csv")

# Fix permissions for the refinery binary
binary_path = Path(nairobi_os.__file__).parent / "bin" / "nairobi-axum-refinery"
if binary_path.exists():
    st = os.stat(binary_path)
    os.chmod(binary_path, st.st_mode | stat.S_IEXEC)
    print(f"✅ Set execution permissions for: {binary_path}")

try:
    # Start the Axum Refinery daemon
    nairobi_os.start_refinery()

    # Run a fused analytics strike
    # (Ingest + Statistics + Correlation in one call)
    result_json = nairobi_os.data.pipeline(
        "dataset.csv",
        "target_col",
        "col1,col2"
    )

    result = json.loads(result_json)

    print(f"\nMean: {result['mean']}")
    print(f"Pearson Correlation: {result['pearson']}")

except Exception as e:
    print(f"❌ Error: {e}")
finally:
    # Stop the refinery
    nairobi_os.stop_refinery()

✅ Created synthetic dataset: dataset.csv
✅ Set execution permissions for: /usr/local/lib/python3.12/dist-packages/nairobi_os/bin/nairobi-axum-refinery

Mean: 0.4526346055428966
Pearson Correlation: 0.00894641630666031


##Real dataset benchmarks

In [5]:
import os
import time
import pandas as pd
import nairobi_os
import json
from pathlib import Path

# 1. Configuration & Data Discovery
data_path = Path(eoinamoore_historical_nba_data_and_player_box_scores_path)
csv_files = list(data_path.glob("**/*.csv"))

# Sort by size to pick substantial datasets for benchmarking
csv_files.sort(key=lambda x: os.path.getsize(x), reverse=True)
target_files = csv_files[:3]  # Benchmark the top 3 largest files

def benchmark_nairobi_vs_pandas(file_path):
    print(f"\n--- Benchmarking: {file_path.name} ({os.path.getsize(file_path)/1024/1024:.2f} MB) ---")

    # Setup features (assuming first col is target, rest are features for the sake of benchmark)
    sample_df = pd.read_csv(file_path, nrows=5)
    cols = sample_df.select_dtypes(include=['number']).columns.tolist()
    if len(cols) < 2:
        print("Skipping: Not enough numeric columns for correlation analysis.")
        return None

    target_col = cols[0]
    feature_cols = ",".join(cols[1:5]) # Take up to 4 features

    # --- PANDAS BENCHMARK ---
    start_p = time.perf_counter()
    df_p = pd.read_csv(file_path)
    p_mean = df_p[target_col].mean()
    p_corr = df_p[target_col].corr(df_p[cols[1]])
    end_p = time.perf_counter()
    pandas_time = end_p - start_p

    # --- NAIROBI BENCHMARK ---
    nairobi_os.start_refinery()
    try:
        start_n = time.perf_counter()
        # Nairobi Fused Pipeline: Load + Stats + Correlation
        res_json = nairobi_os.data.pipeline(str(file_path), target_col, feature_cols)
        res = json.loads(res_json)
        end_n = time.perf_counter()
        nairobi_time = end_n - start_n
    finally:
        nairobi_os.stop_refinery()

    print(f"Pandas Time:  {pandas_time:.4f}s (Mean: {p_mean:.2f})")
    print(f"Nairobi Time: {nairobi_time:.4f}s (Mean: {res['mean']:.2f})")

    speedup = pandas_time / nairobi_time
    print(f"🚀 Nairobi is {speedup:.2f}x faster")

    return {"file": file_path.name, "pandas": pandas_time, "nairobi": nairobi_time, "speedup": speedup}

results = []
for f in target_files:
    res = benchmark_nairobi_vs_pandas(f)
    if res: results.append(res)

# Summary Table
if results:
    print("\n" + "="*40)
    print("FINAL BENCHMARK SUMMARY")
    print("="*40)
    summary_df = pd.DataFrame(results)
    print(summary_df)
else:
    print("No valid numeric data found for benchmarking.")


--- Benchmarking: PlayerStatisticsExtended.csv (431.96 MB) ---


/tmp/ipykernel_8486/2014322053.py:31: DtypeWarning: Columns (4,6,7,8,13,15,16,17,18,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df_p = pd.read_csv(file_path)


Pandas Time:  13.6093s (Mean: 1163748.05)
Nairobi Time: 5.4191s (Mean: 1163748.05)
🚀 Nairobi is 2.51x faster

--- Benchmarking: PlayerStatistics.csv (371.44 MB) ---


/tmp/ipykernel_8486/2014322053.py:31: DtypeWarning: Columns (10,11,12,15,37,38) have mixed types. Specify dtype option on import or set low_memory=False.
  df_p = pd.read_csv(file_path)


Pandas Time:  9.1983s (Mean: 734735.58)
Nairobi Time: 4.8998s (Mean: 734735.58)
🚀 Nairobi is 1.88x faster

--- Benchmarking: TeamStatisticsExtended.csv (36.39 MB) ---
Pandas Time:  0.8829s (Mean: 23012868.66)
Nairobi Time: 0.6862s (Mean: 23012868.66)
🚀 Nairobi is 1.29x faster

FINAL BENCHMARK SUMMARY
                           file     pandas   nairobi   speedup
0  PlayerStatisticsExtended.csv  13.609302  5.419138  2.511341
1          PlayerStatistics.csv   9.198284  4.899848  1.877259
2    TeamStatisticsExtended.csv   0.882885  0.686228  1.286576


##Cross CSV benchmarks

In [7]:
import time
import pandas as pd
import nairobi_os
import json
import os
from pathlib import Path

# Use the top 3 files identified earlier
target_files = [str(f) for f in csv_files[:3]]

def multi_file_benchmark():
    print(f"--- Multi-File Benchmark (Ingesting {len(target_files)} CSVs) ---")

    # 1. Determine columns for each file to avoid 'no columns' errors
    file_configs = []
    for f in target_files:
        sample = pd.read_csv(f, nrows=1)
        num_cols = sample.select_dtypes(include=['number']).columns.tolist()
        # Pick a target and a feature present in the file
        target = num_cols[0] if num_cols else None
        features = ",".join(num_cols[1:3]) if len(num_cols) > 2 else ""
        file_configs.append({'path': f, 'target': target, 'features': features, 'cols': num_cols[:3]})

    # --- PANDAS APPROACH ---
    # Simulating a batch processing task: Read all files and describe them
    start_p = time.perf_counter()
    for config in file_configs:
        df_temp = pd.read_csv(config['path'], usecols=config['cols'])
        _ = df_temp.describe()
    pandas_total = time.perf_counter() - start_p

    # --- NAIROBI APPROACH ---
    nairobi_os.start_refinery()
    try:
        start_n = time.perf_counter()
        # Process files in a batch strike
        for config in file_configs:
            if config['target']:
                res_json = nairobi_os.data.pipeline(config['path'], config['target'], config['features'])
                _ = json.loads(res_json)
        nairobi_total = time.perf_counter() - start_n
    finally:
        nairobi_os.stop_refinery()

    print(f"Pandas Total Batch Time:  {pandas_total:.4f}s")
    print(f"Nairobi Total Batch Time: {nairobi_total:.4f}s")

    if nairobi_total > 0:
        print(f"🚀 Speedup: {pandas_total/nairobi_total:.2f}x faster")
    else:
        print("Benchmark finished too quickly to measure.")

multi_file_benchmark()

--- Multi-File Benchmark (Ingesting 3 CSVs) ---


/tmp/ipykernel_8486/1416700965.py:28: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(config['path'], usecols=config['cols'])
/tmp/ipykernel_8486/1416700965.py:28: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(config['path'], usecols=config['cols'])


Pandas Total Batch Time:  8.7689s
Nairobi Total Batch Time: 11.8347s
🚀 Speedup: 0.74x faster


In [9]:
import psutil
import os
import gc
import time
import pandas as pd
import nairobi_os
import json
from pathlib import Path

def get_memory_usage():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / (1024 * 1024)

# Using the second largest file
test_file = target_files[1]

# Pre-detect numeric columns to avoid TypeErrors
sample = pd.read_csv(test_file, nrows=100)
num_cols = sample.select_dtypes(include=['number']).columns.tolist()
target_col = num_cols[0]

print(f"--- Memory Usage Benchmark: {Path(test_file).name} ---")
print(f"Target Column: {target_col}")

# --- PANDAS MEMORY ---
gc.collect()
mem_before_p = get_memory_usage()
df_p = pd.read_csv(test_file, low_memory=False)
_ = df_p[target_col].mean()
mem_after_p = get_memory_usage()
pandas_mem_delta = mem_after_p - mem_before_p
print(f"Pandas Peak Memory Increase: {pandas_mem_delta:.2f} MB")

del df_p
gc.collect()

# --- NAIROBI MEMORY ---
gc.collect()
mem_before_n = get_memory_usage()
nairobi_os.start_refinery()
try:
    res_json = nairobi_os.data.pipeline(str(test_file), target_col, "")
    mem_after_n = get_memory_usage()
    nairobi_mem_delta = mem_after_n - mem_before_n
    print(f"Nairobi (Python-side) Memory Increase: {nairobi_mem_delta:.2f} MB")
finally:
    nairobi_os.stop_refinery()

print(f"\n✅ Memory Savings in Python Process: {pandas_mem_delta - nairobi_mem_delta:.2f} MB")

--- Memory Usage Benchmark: PlayerStatistics.csv ---
Target Column: personId
Pandas Peak Memory Increase: 526.32 MB
Nairobi (Python-side) Memory Increase: 0.00 MB

✅ Memory Savings in Python Process: 526.32 MB


In [10]:
import psutil
import time
import nairobi_os
import pandas as pd
from pathlib import Path

def get_daemon_memory():
    """Finds the nairobi-axum-refinery process and returns its memory in MB."""
    for proc in psutil.process_iter(['name']):
        try:
            if 'nairobi-axum' in proc.info['name']:
                return proc.memory_info().rss / (1024 * 1024)
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            pass
    return 0.0

test_file = target_files[1]
print(f"--- Full System Memory Benchmark: {Path(test_file).name} ---")

nairobi_os.start_refinery()
time.sleep(1) # Let the daemon settle

try:
    base_mem = get_daemon_memory()
    print(f"Daemon Base Memory (Idle): {base_mem:.2f} MB")

    # Run the pipeline and capture peak memory
    # Note: In a real environment, we'd poll this in a thread, but for a
    # single large file, we can check immediately after the call starts.
    n_res_json = nairobi_os.data.pipeline(str(test_file), "personId", "")

    peak_mem = get_daemon_memory()
    print(f"Daemon Peak Memory (Active): {peak_mem:.2f} MB")
    print(f"\n✅ Total Nairobi System Overhead: {peak_mem:.2f} MB")
    print(f"✅ Total Pandas Memory for same task: {pandas_mem_delta:.2f} MB")

    savings = pandas_mem_delta - peak_mem
    print(f"\n⌒️ Net Hardware Savings: {savings:.2f} MB")

finally:
    nairobi_os.stop_refinery()

--- Full System Memory Benchmark: PlayerStatistics.csv ---
Daemon Base Memory (Idle): 9.11 MB
Daemon Peak Memory (Active): 417.87 MB

✅ Total Nairobi System Overhead: 417.87 MB
✅ Total Pandas Memory for same task: 526.32 MB

⌒️ Net Hardware Savings: 108.45 MB
